### IMPORT REQUIREMENTS

In [17]:
import time
import sys
import cv2
import os
import yaml
import torch
import numpy as np
from PIL import Image
from pathlib import Path
import tensorrt as trt
from ultralytics import YOLO
import matplotlib.pyplot as plt
from torchvision import transforms
import torchvision.transforms as transforms
from cuda.bindings import runtime as cudart
import shutil
from zipfile import ZipFile


# PHASE 1

# Yolo .pt Model


### LOAD .pt MODEL & IMAGE FOLDER


In [18]:
# Path of Your Project  Dir.
project_dir=os.getcwd()
print(project_dir)

/home/easemyai/Downloads/yolo_inference_evaluation/Detection_Model_Evaluation


### Download YOLO Model

In [19]:
# Download yolo Model

# You Can Change Model Name According To Your Requirement
model_name = "yolo26n.pt"
model = YOLO(model_name)

m_load=os.path.join(os.getcwd(),model_name)

# Moveing Model In To Model Directory 
directory_path=os.path.join(project_dir,"Detection_Models/")
full_path_model=os.path.join(directory_path,model_name)

shutil.move(m_load,full_path_model)

'/home/easemyai/Downloads/yolo_inference_evaluation/Detection_Model_Evaluation/Detection_Models/yolo26n.pt'

### **Download Sample Images From Below Given Link or You Can Use Yours As Well**
##### https://drive.google.com/drive/folders/1MuNF6ytZHTcroBpnlwIUoWkaxl0oTTOS

In [20]:
# Change Zip File Path According To Your Zip Path

Downloaded_dir="/home/easemyai/Downloads/"
zip_name="Images-20260709T083605Z-3-001.zip"

images_path=os.path.join(Downloaded_dir,zip_name)
with ZipFile(images_path, 'r') as zObject:
    zObject.extractall(path=project_dir)

#### YOLO object detection models are trained on the COCO dataset. During inference, Ultralytics automatically maps the predicted class indices to their corresponding COCO class names using the dataset configuration.

#### The COCO configuration is not required for Phase 1, but it is required for Phase 2 and Phase 3.

#### The COCO dataset configuration is available here:

https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/datasets/coco.yaml

In [21]:
# Load yaml file into your cls-yaml directory
def find(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)

# Update This Downloaded Path According To Your Path
Downloaded_dir="/home/easemyai/Downloads/"

YAML_name="coco.yaml"

full_path_yaml=os.path.join(project_dir,"detection-yaml")
path_yaml_file=find(YAML_name,Downloaded_dir)
shutil.move(path_yaml_file,full_path_yaml)

'/home/easemyai/Downloads/yolo_inference_evaluation/Detection_Model_Evaluation/detection-yaml/coco.yaml'

In [22]:
# Accessing Model, Images, Yaml from your Project Dir.
model_pt = "Detection_Models/yolo26n.pt"
images = "Images"
coco="detection-yaml/coco.yaml"

model_path_pt=os.path.join(os.getcwd(),model_pt)
image_folder=os.path.join(os.getcwd(),images)
coco_yaml=os.path.join(os.getcwd(),coco)
print("Model Path: ",model_path_pt)
print("Image Folder Path: ", image_folder)
print("Imagenet YAML Path: ",coco_yaml)

# Image Size
img_size = 640

Model Path:  /home/easemyai/Downloads/yolo_inference_evaluation/Detection_Model_Evaluation/Detection_Models/yolo26n.pt
Image Folder Path:  /home/easemyai/Downloads/yolo_inference_evaluation/Detection_Model_Evaluation/Images
Imagenet YAML Path:  /home/easemyai/Downloads/yolo_inference_evaluation/Detection_Model_Evaluation/detection-yaml/coco.yaml


### LOAD MODEL TO GPU

In [ ]:
# Load Model Into GPU

device = torch.device("cuda")

# device="cuda"

model = YOLO(model_path_pt)

model.to(device)

### PREPROCESS

In [ ]:
# Yolo Detection models are trained with an input image size of 640 × 640.

# Although the original image can have any resolution, it will be resized to 640 × 640 during preprocessing before being passed to the model.
# The preprocessing pipeline includes operations such as:

# - Resize
# - ToTensor
# - Normalize

# Batch creation is important because neural networks expect the input in the form:

# Batch Size × Channels × Height × Width

# (B, C, H, W)

# instead of processing one image at a time.


def preprocess(folder_path):

    image_paths = sorted([
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        )
    ])

    print("=" * 60)
    print("PREPROCESS")
    print("=" * 60)

    print(f"Number of Images : {len(image_paths)}")

    for path in image_paths:
        print(os.path.basename(path))

    return image_paths

### INFERENCE

In [ ]:
# During this stage, the preprocessed batch is passed to the model.

# The GPU performs the forward pass and generates prediction for every image in the batch.


def inference(model, folder_path):

    torch.cuda.synchronize()

    start = time.perf_counter()

    results = model.predict(
        source=folder_path,
        imgsz=640,
        device=0,
        verbose=False
    )

    torch.cuda.synchronize()

    end = time.perf_counter()

    inference_time = (end - start) * 1000

    batch_size = len(results)

    print("\n" + "=" * 60)
    print("PYTORCH DETECTION PERFORMANCE")
    print("=" * 60)

    print(f"Batch Size           : {batch_size}")
    print(f"Batch Inference Time : {inference_time:.3f} ms")
    print(f"Per Image Latency    : {inference_time/batch_size:.3f} ms")
    print(f"Throughput           : {batch_size/(end-start):.2f} images/sec")

    return results


### POSTPROCESS

In [ ]:
# This stage converts the raw model outputs into human-readable predictions.

# Operations include:
# - Extract prediction score (confidence)
# - Find the class with the highest confidence
# - Display predicted class name
# - Display confidence score
def postprocess(results, image_paths):

    print("\n" + "=" * 60)
    print("POSTPROCESS")
    print("=" * 60)

    for result, image_path in zip(results, image_paths):

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if len(result.boxes) == 0:

            print(f"\nImage : {os.path.basename(image_path)}")
            print("No Detection")

            plt.figure(figsize=(8,6))
            plt.imshow(image)
            plt.axis("off")
            plt.show()

            continue

        confidences = result.boxes.conf

        top_idx = confidences.argmax()

        box = result.boxes[top_idx]

        class_id = int(box.cls.item())

        class_name = result.names[class_id]

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (255, 0, 0),
            2
        )

        cv2.putText(
            image,
            f"{class_name} {confidence:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 0, 0),
            2
        )

        print(f"\nImage      : {os.path.basename(image_path)}")
        print(f"Prediction : {class_name}")
        print(f"Confidence : {confidence:.4f}")

        plt.figure(figsize=(8,6))
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"{class_name} ({confidence:.2f})")
        plt.show()

### Main

In [ ]:
# Calling All Functions 
image_paths = preprocess(image_folder)
results = inference(model,image_folder)
postprocess(results,image_paths)

# Phase 2

## ONNX

In [ ]:
pip install onnxruntime-gpu --break-system-packages

### EXPORT TO ONNX

In [ ]:
import onnxruntime as ort

# Genarating onnx model using yolo .pt model file and load into cuda GPU 

model.export(format="onnx",imgsz=640,batch=16,dynamic=True,device="cuda")

print("\nONNX export completed successfully.")

### LOAD ONNX MODEL PATH

In [ ]:
onnx_model = os.path.join(os.getcwd(),"Detection_Models/yolo26n.onnx")
print(onnx_model)

##### But All Available Executor you cannot use. At the Beginning if 'CUDAExecutionProvider' will show in available but not in session
##### If Using Provider only shows ['CPUExecutionProvider'] 
##### So For That You have install cuda version and other requirements according to you onnxruntime version. Follow Steps present in INSTALLATION.md
##### To see which Executor you can use for session execute following script.

In [ ]:
# Loading Your Session into GPU
# If session by default load into CPUExecutionProvider then you CUDAExecutionProvider not in work for execution.
# So follow the steps present in the INSTALLATION.md to solve this problem 

opts = ort.SessionOptions()

# Force ONNX Runtime to raise an exception if an op cannot run on the requested EP

opts.add_session_config_entry("session.required_cuda_compute_capability", "0") 

# This configuration forces strict provider matching
session = ort.InferenceSession(onnx_model, providers=['CUDAExecutionProvider'], sess_options=opts)

input_name = session.get_inputs()[0].name

print("Strict session created successfully on GPU!")
print("Available Providers:", ort.get_available_providers())
print("Session Providers:", session.get_providers())


### PREPROCESS

In [ ]:
# Preprocess Phase 
def preprocess_onnx(folder_path, imgsz=640):

    transform = transforms.Compose([transforms.Resize((imgsz, imgsz)),transforms.ToTensor()])

    image_files = sorted([
        file
        for file in os.listdir(folder_path)
        if file.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        )
    ])

    image_list = []

    print("=" * 60)
    print("PREPROCESS")
    print("=" * 60)

    for file in image_files:

        image_path = os.path.join(folder_path, file)

        image = Image.open(image_path).convert("RGB")

        tensor = transform(image)

        image_list.append(tensor)

    batch_tensor = torch.stack(image_list)

    batch_numpy = batch_tensor.numpy().astype(np.float32)

    print("Number of Images :", len(image_files))
    print("Batch Shape      :", batch_numpy.shape)
    print("Input dtype      :", batch_numpy.dtype)
    print()

    return batch_numpy, image_files

### INFERENCE

In [ ]:
# Inference Phase
def inference_onnx(session,input_name, batch_numpy):


    start = time.perf_counter()

    outputs = session.run(None,{input_name: batch_numpy})

    end = time.perf_counter()

    # inference_time = (end - start) * 1000
    inference_time = (end - start)

    batch_size = batch_numpy.shape[0]

    print("=" * 60)
    print("ONNX RUNTIME PERFORMANCE")
    print("=" * 60)
    print(f"Batch Size           : {batch_size}")
    print(f"Batch Inference Time : {inference_time:.3f}")
    print(f"Per Image Latency    : {inference_time / batch_size:.3f}")
    print()

    return outputs[0], inference_time

### POSTPROCESS

In [ ]:
def postprocess_onnx(outputs,image_files,image_folder,yaml_path,confidence_threshold=0.6,input_size=640):

    if isinstance(input_size, int):
        input_size = (input_size, input_size)

    input_h, input_w = input_size

    # Load class names
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    class_names = data["names"]

    if isinstance(class_names, dict):
    
        names = []
    
        for key in sorted(class_names.keys()):
            names.append(class_names[key])
    
        class_names = names

 
    for img_idx, image_name in enumerate(image_files):

        image_path = os.path.join(image_folder, image_name)

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        H, W = image.shape[:2]

        scale_x = W / input_w
        scale_y = H / input_h

        detections = outputs[img_idx]

        for det in detections:

            x1, y1, x2, y2, score, cls = det

            if score < confidence_threshold:
                continue

            cls = int(cls)

            x1 = int(x1 * scale_x)
            y1 = int(y1 * scale_y)
            x2 = int(x2 * scale_x)
            y2 = int(y2 * scale_y)

            x1 = max(0, min(x1, W - 1))
            y1 = max(0, min(y1, H - 1))
            x2 = max(0, min(x2, W - 1))
            y2 = max(0, min(y2, H - 1))

            label = f"{class_names[cls]} {score:.2f}"

            cv2.rectangle(image,(x1, y1),(x2, y2),(0, 255, 0),2)

            cv2.putText(image,label,(x1, max(20, y1 - 10)),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255, 0, 0),2)

        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        plt.title(image_name)
        plt.axis("off")
        plt.show()

### MAIN

In [ ]:
def main():
    # Preprocess
    batch_numpy, image_files = preprocess_onnx(image_folder,img_size)

    # Inference
    predictions,inference_time = inference_onnx(session,input_name,batch_numpy)

    # Postprocess
    postprocess_onnx(predictions,image_files,image_folder,coco_yaml,confidence_threshold=0.6,input_size=img_size)

if __name__=="__main__":
    main()

# Phase 3

# TensorRT Engine Model

##### Go To Your Terminal Inside Your Project Directory And Execute This Following 2 Commands

##### 1)
###### cd Detection_Model_Evaluation/Detection_Models

##### 2)
###### trtexec \
###### ---onnx=yolo26n.onnx \
###### --minShapes=images:1x3x640x640 \
###### --optShapes=images:4x3x640x640 \
###### --maxShapes=images:16x3x640x640 \
###### --saveEngine=yolo26n.engine


#### LOAD ENGINE DEL PATH

In [ ]:
engine_model=os.path.join(os.getcwd(),"Detection_Models/yolo26n.engine")
print(engine_model)

### Check Error Function 

In [ ]:
# Check Function for Checking error logs
def check_cuda(err):

    if isinstance(err,tuple):
        err=err[0]

    if err != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"CUDA Error : {err}")

### PREPROCESS

In [ ]:
def preprocess_engine(image_folder, input_size=640):


    # Convert integer to (H, W)

    input_size = (input_size, input_size)

    transform = transforms.Compose([transforms.Resize(size=input_size),transforms.ToTensor()])

    # Read image filenames
    image_files = sorted([
        file for file in os.listdir(image_folder)
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))])

    if len(image_files) == 0:
        raise ValueError("No images found in the specified folder.")

    batch_images = []

    for image_name in image_files:

        image_path = os.path.join(image_folder, image_name)

        image = Image.open(image_path).convert("RGB")

        tensor = transform(image)

        batch_images.append(tensor)

    # Stack into batch
    batch_tensor = torch.stack(batch_images, dim=0)

    # Convert to NumPy
    batch_numpy = batch_tensor.numpy().astype(np.float32)

    print(f"Number of Images : {len(image_files)}")
    print(f"Input Shape      : {batch_numpy.shape}")

    return batch_numpy, image_files

### INFERENCE

In [ ]:
def inference_engine(batch_numpy,engine_model):


    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

    with open(engine_model, "rb") as f:
        runtime = trt.Runtime(TRT_LOGGER)
        engine = runtime.deserialize_cuda_engine(f.read())

    context = engine.create_execution_context()
    
    input_name = engine.get_tensor_name(0)
    
    output_name = engine.get_tensor_name(1)
    

    context.set_input_shape(input_name, batch_numpy.shape)

    output_shape = tuple(context.get_tensor_shape(output_name))

    host_input = np.ascontiguousarray(batch_numpy)
    host_output = np.empty(output_shape, dtype=np.float32)

    # Allocate GPU memory
    err, d_input = cudart.cudaMalloc(host_input.nbytes)
    check_cuda(err)

    err, d_output = cudart.cudaMalloc(host_output.nbytes)
    check_cuda(err)

    # Create CUDA stream
    err, stream = cudart.cudaStreamCreate()
    check_cuda(err)


    start = time.perf_counter()

    # Host -> Device
    check_cuda(
        cudart.cudaMemcpyAsync(
            d_input,
            host_input.ctypes.data,
            host_input.nbytes,
            cudart.cudaMemcpyKind.cudaMemcpyHostToDevice,
            stream,
        )
    )

    # Bind tensors
    context.set_tensor_address(input_name, int(d_input))
    context.set_tensor_address(output_name, int(d_output))

    # TensorRT inference
    context.execute_async_v3(stream)

    # Device -> Host
    check_cuda(
        cudart.cudaMemcpyAsync(
            host_output.ctypes.data,
            d_output,
            host_output.nbytes,
            cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost,
            stream,
        )
    )

    # Wait for GPU
    check_cuda(cudart.cudaStreamSynchronize(stream))

    # -----------------------------
    # End Timer
    # -----------------------------
    end = time.perf_counter()

    # inference_time = (end - start) * 1000
    inference_time = (end - start)

    batch_size = batch_numpy.shape[0]

    print("\n" + "=" * 60)
    print("TENSORRT PERFORMANCE")
    print("=" * 60)
    print(f"Batch Size           : {batch_size}")
    print(f"Batch Inference Time : {inference_time:.3f} ")
    print(f"Per Image Latency    : {inference_time/batch_size:.3f} ")


    # Cleanup
    cudart.cudaFree(d_input)
    cudart.cudaFree(d_output)
    cudart.cudaStreamDestroy(stream)

    return host_output

### POSTPROCESS

In [ ]:
def postprocess_engine(outputs,image_files,image_folder,yaml_path,confidence_threshold=0.6,input_size=640):

    if isinstance(input_size, int):
        input_size = (input_size, input_size)

    input_h, input_w = input_size

    # Load class names
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    class_names = data["names"]

    if isinstance(class_names, dict):
    
        names = []
    
        for key in sorted(class_names.keys()):
            names.append(class_names[key])
    
        class_names = names

 
    for img_idx, image_name in enumerate(image_files):

        image_path = os.path.join(image_folder, image_name)

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        H, W = image.shape[:2]

        scale_x = W / input_w
        scale_y = H / input_h

        detections = outputs[img_idx]

        for det in detections:

            x1, y1, x2, y2, score, cls = det

            if score < confidence_threshold:
                continue

            cls = int(cls)

            x1 = int(x1 * scale_x)
            y1 = int(y1 * scale_y)
            x2 = int(x2 * scale_x)
            y2 = int(y2 * scale_y)

            x1 = max(0, min(x1, W - 1))
            y1 = max(0, min(y1, H - 1))
            x2 = max(0, min(x2, W - 1))
            y2 = max(0, min(y2, H - 1))

            label = f"{class_names[cls]} {score:.2f}"

            cv2.rectangle(image,(x1, y1),(x2, y2),(0, 255, 0),2)

            cv2.putText(image,label,(x1, max(20, y1 - 10)),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255, 0, 0),2)

        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        plt.title(image_name)
        plt.axis("off")
        plt.show()

### MAIN

In [ ]:
def main():
    # Preprocess
    batch_numpy, image_files = preprocess_engine(image_folder,img_size)

    # Inference
    predictions= inference_engine(batch_numpy,engine_model)

    # Postprocess
    postprocess_engine(predictions,image_files,image_folder,coco_yaml,confidence_threshold=0.6,input_size=img_size)

if __name__=="__main__":
    main()